# bug-report-checker: fine-tune Laya on Kaggle

Settings: **Accelerator GPU T4**, **Internet On**. Add a private dataset with `train_v2.jsonl` (from `python -m bug_report_checker.dataset.laya_format`).

Training is `laya.train.finetune` (what `laya-train` runs) from the pinned `laya` package: the published notebook's loop and RLCD objective, one GPU. laya 0.4.0 evaluates the base model before moving it to the GPU and fails on CUDA, with or without `--eval`, so the checkpoint is moved right after loading. The eval set is not passed: runs are compared on a separate dev set and eval is scored once, on the chosen model (D13). The fine-tuned checkpoint is written to `/kaggle/working/bug-report-checker-v2` — download it from the notebook output.

In [ ]:
!pip install -q laya==0.4.0

In [ ]:
import glob
import os

from huggingface_hub import snapshot_download

# Kaggle's mount path for an input dataset varies; find the files instead of guessing.
[TRAIN] = glob.glob("/kaggle/input/**/train_v2.jsonl", recursive=True)
DATA = os.path.dirname(TRAIN)
print(DATA)
OUT = "/kaggle/working/bug-report-checker-v2"
# The base checkpoint at the revision the local dry-run and baseline used.
REVISION = "7b928d828b7b0e022f929d9bd2e44165aa270148"
BASE = os.path.join(
    snapshot_download(
        "convaiinnovations/laya", revision=REVISION, allow_patterns="typed-decisions/*"
    ),
    "typed-decisions",
)
print(BASE)

In [ ]:
import laya.train as lt

_load_checkpoint = lt.load_checkpoint


def load_on_device(model_dir):
    # laya 0.4.0 runs the before-training eval while the model is still on the CPU.
    model, tok, cfg = _load_checkpoint(model_dir)
    return model.to(lt.resolve_device("auto")), tok, cfg


lt.load_checkpoint = load_on_device
summary = lt.finetune(
    f"{DATA}/train_v2.jsonl",
    BASE,
    OUT,
    lt.TrainConfig(seed=0),
)

In [ ]:
%cd /kaggle/working
!zip -qr bug-report-checker-v2.zip bug-report-checker-v2 -x '*/checkpoint_latest/*'
!ls -lh